## Climate Zones - Data Preparation
* Author: Stephen Haddad
* Affiliation: UK Met Office
* History: 1.0
* Last update: 2026-03-16
* © British Crown Copyright 2017-2026, Met Office. Please see LICENSE.md for license details.

This notebook is not a core part of the tutorial, but rather to document the process of how the dataset was prepared for the tutorial from the original dataset.

In [2]:
import pathlib
import os

In [3]:
import matplotlib.pyplot
import cartopy.crs

In [39]:
import xarray
import pandas

#### Dataset parameters

In [5]:
def get_platform_dir(select_platform):
    if select_platform == 'mo_linux':
      return pathlib.Path(os.environ['DATADIR']) / 'climate_zones'
    if select_platform == 'jasmin':
        return pathlib.Path('/gws/nopw/j04/mohc_shared/dscop/') / 'climate_zones'
    print('platform not found, return generic path')
    return pathlib.Path(os.environ['HOME']) / 'climate_zones',

current_platform = 'mo_linux'

In [6]:
root_data_dir = get_platform_dir(current_platform)

print(root_data_dir.is_dir())
root_data_dir

True


PosixPath('/data/users/stephen.haddad/climate_zones')

In [7]:
ml_ready_output_dir = root_data_dir / 'ml_ready'

if ml_ready_output_dir.is_dir():
    print(f'output directory {str(ml_ready_output_dir)} exists')
else:
    ml_ready_output_dir.mkdir(parents=False)
    print(f'created output directory {str(ml_ready_output_dir)}')    
    

output directory /data/users/stephen.haddad/climate_zones/ml_ready exists


In [8]:
resolutions_dict = {
    0.1: '0p1',
    0.5: '0p5',
    1.0: '1p0',
}

dataset_prefix_dict = {
    'climate_mean': 'ensemble_mean',
    'climate_std': 'ensemble_std',
    'climate_zone': 'koppen_geiger',
    
}
format_str = 'nc'
historic_scenario_str = 'historic'

future_scenario_list = ['ssp119','ssp126','ssp245','ssp370','ssp434','ssp460','ssp585']
historic_scenario_list = [historic_scenario_str,]

time_periods = { 
    (1901,1930): historic_scenario_list, 
    (1931,1960): historic_scenario_list,
    (1961,1990): historic_scenario_list,
    (1991,2020): historic_scenario_list,
    (2041,2070): future_scenario_list,
    (2071,2099): future_scenario_list,
}


In [45]:
fname_template = '{prefix}_{res}.{suffix}'
time_dir_template = '{start_year}_{end_year}'
csv_out_template = 'climate_zones_{resolution}.csv'

The following lables for climate zones are taken from the `legends.txt` file in the [dataset](https://figshare.com/articles/dataset/High-resolution_1_km_K_ppen-Geiger_maps_for_1901_2099_based_on_constrained_CMIP6_projections/21789074)

In [10]:
legend_path = root_data_dir / 'legend.txt'
with open(legend_path) as f1:
    zones_legend = f1.readlines()
print(''.join(zones_legend))


Legend linking the numeric values in the maps to the Köppen-Geiger classes.
The RGB colors used in Beck et al. (2023) are provided between brackets.

    1:  Af   Tropical, rainforest                  [0 0 255]
    2:  Am   Tropical, monsoon                     [0 120 255]
    3:  Aw   Tropical, savannah                    [70 170 250]
    4:  BWh  Arid, desert, hot                     [255 0 0]
    5:  BWk  Arid, desert, cold                    [255 150 150]
    6:  BSh  Arid, steppe, hot                     [245 165 0]
    7:  BSk  Arid, steppe, cold                    [255 220 100]
    8:  Csa  Temperate, dry summer, hot summer     [255 255 0]
    9:  Csb  Temperate, dry summer, warm summer    [200 200 0]
    10: Csc  Temperate, dry summer, cold summer    [150 150 0]
    11: Cwa  Temperate, dry winter, hot summer     [150 255 150]
    12: Cwb  Temperate, dry winter, warm summer    [100 200 100]
    13: Cwc  Temperate, dry winter, cold summer    [50 150 50]
    14: Cfa  Temperate, no

In [11]:
climate_groups_dict = dict([
    ('non-land','None'), #this corresponds to the zero in the data. Will be filtered out of training data.
    ('Af','Tropical, rainforest'),
    ('Am','Tropical, monsoon'),
    ('Aw','Tropical, savannah'),
    ('BWh','Arid, desert, hot'),
    ('BWk','Arid, desert, cold'),
    ('BSh','Arid, steppe, hot'),
    ('BSk','Arid, steppe, cold'),
    ('Csa','Temperate, dry summer, hot summer'),
    ('Csb','Temperate, dry summer, warm summer'),
    ('Csc','Temperate, dry summer, cold summer'),
    ('Cwa','Temperate, dry winter, hot summer'),
    ('Cwb','Temperate, dry winter, warm summer'),
    ('Cwc','Temperate, dry winter, cold summer'),
    ('Cfa','Temperate, no dry season, hot summer'),
    ('Cfb','Temperate, no dry season, warm summer'),
    ('Cfc','Temperate, no dry season, cold summer'),
    ('Dsa','Cold, dry summer, hot summer'),
    ('Dsb','Cold, dry summer, warm summer'),
    ('Dsc','Cold, dry summer, cold summer'),
    ('Dsd','Cold, dry summer, very cold winter'),
    ('Dwa','Cold, dry winter, hot summer'),
    ('Dwb','Cold, dry winter, warm summer'),
    ('Dwc','Cold, dry winter, cold summer'),
    ('Dwd','Cold, dry winter, very cold winter'),
    ('Dfa','Cold, no dry season, hot summer'),
    ('Dfb','Cold, no dry season, warm summer'),
    ('Dfc','Cold, no dry season, cold summer'),
    ('Dfd',',Cold, no dry season, very cold winter'),
    ('ET','Polar, tundra'),
    ('EF','Polar, frost'),
])

In [12]:
climate_subgroup_lookup = [k1 for k1 in climate_groups_dict.keys()]

In [13]:
climate_subgroup_lookup

['non-land',
 'Af',
 'Am',
 'Aw',
 'BWh',
 'BWk',
 'BSh',
 'BSk',
 'Csa',
 'Csb',
 'Csc',
 'Cwa',
 'Cwb',
 'Cwc',
 'Cfa',
 'Cfb',
 'Cfc',
 'Dsa',
 'Dsb',
 'Dsc',
 'Dsd',
 'Dwa',
 'Dwb',
 'Dwc',
 'Dwd',
 'Dfa',
 'Dfb',
 'Dfc',
 'Dfd',
 'ET',
 'EF']

For comparison we will try predicting all 30 zones, but also trying to predict the five zone groupings for a simpler problem.

In [14]:
major_zones_dict = {
    'A': range(1,4),
    'B': range(4,8),
    'C': range(8,17),
    'D': range(17,29),
    'E': range(29,31),
}

In [15]:
climate_group_lookup = [0 for i1 in range (31)]
for ix1, (grouping, group_indices) in enumerate(major_zones_dict.items()):
    for zone_index in group_indices:
        climate_group_lookup[zone_index] = grouping
climate_group_lookup[0] = 'none'        

In [16]:
climate_group_lookup

['none',
 'A',
 'A',
 'A',
 'B',
 'B',
 'B',
 'B',
 'C',
 'C',
 'C',
 'C',
 'C',
 'C',
 'C',
 'C',
 'C',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'E',
 'E']

### Load the data

In [17]:
def get_data_path(root_dir, time_period, scenario_id, prefix, resolution, suffix) :
    start_year = time_period[0]
    end_year = time_period[1]
    if scenario_id == historic_scenario_str:
        data_dir = root_data_dir / time_dir_template.format(start_year=start_year,end_year=end_year)
    else:
        data_dir = root_data_dir / time_dir_template.format(start_year=start_year,end_year=end_year) / scenario_id 
    data_fname = fname_template.format(prefix=prefix, 
                                       res=resolutions_dict[current_res], 
                                       suffix=format_str)
    return data_dir / data_fname

In [18]:
current_res = 1.0

In [19]:
data_path_dict = {
    (start_year, end_year): {
        scenario_id: { ds_id: get_data_path(root_dir=root_data_dir,
                                            time_period=(start_year, end_year),
                                            scenario_id=scenario_id, 
                                            prefix=ds_str,
                                            resolution=current_res,
                                            suffix=format_str,
                                           )
                       for ds_id, ds_str in dataset_prefix_dict.items()
                     } 
        for scenario_id in current_scenarios
    }
    for (start_year, end_year), current_scenarios in time_periods.items()                                                                                                             
}

In [20]:
for cp, ps in data_path_dict.items():
    for sid, pd in ps.items():
        for ds_id, ds_path in pd.items():
            print(cp, ds_id, ds_path.is_file())
data_path_dict

(1901, 1930) climate_mean True
(1901, 1930) climate_std True
(1901, 1930) climate_zone True
(1931, 1960) climate_mean True
(1931, 1960) climate_std True
(1931, 1960) climate_zone True
(1961, 1990) climate_mean True
(1961, 1990) climate_std True
(1961, 1990) climate_zone True
(1991, 2020) climate_mean True
(1991, 2020) climate_std True
(1991, 2020) climate_zone True
(2041, 2070) climate_mean True
(2041, 2070) climate_std True
(2041, 2070) climate_zone True
(2041, 2070) climate_mean True
(2041, 2070) climate_std True
(2041, 2070) climate_zone True
(2041, 2070) climate_mean True
(2041, 2070) climate_std True
(2041, 2070) climate_zone True
(2041, 2070) climate_mean True
(2041, 2070) climate_std True
(2041, 2070) climate_zone True
(2041, 2070) climate_mean True
(2041, 2070) climate_std True
(2041, 2070) climate_zone True
(2041, 2070) climate_mean True
(2041, 2070) climate_std True
(2041, 2070) climate_zone True
(2041, 2070) climate_mean True
(2041, 2070) climate_std True
(2041, 2070) climat

{(1901,
  1930): {'historic': {'climate_mean': PosixPath('/data/users/stephen.haddad/climate_zones/1901_1930/ensemble_mean_1p0.nc'),
   'climate_std': PosixPath('/data/users/stephen.haddad/climate_zones/1901_1930/ensemble_std_1p0.nc'),
   'climate_zone': PosixPath('/data/users/stephen.haddad/climate_zones/1901_1930/koppen_geiger_1p0.nc')}},
 (1931,
  1960): {'historic': {'climate_mean': PosixPath('/data/users/stephen.haddad/climate_zones/1931_1960/ensemble_mean_1p0.nc'),
   'climate_std': PosixPath('/data/users/stephen.haddad/climate_zones/1931_1960/ensemble_std_1p0.nc'),
   'climate_zone': PosixPath('/data/users/stephen.haddad/climate_zones/1931_1960/koppen_geiger_1p0.nc')}},
 (1961,
  1990): {'historic': {'climate_mean': PosixPath('/data/users/stephen.haddad/climate_zones/1961_1990/ensemble_mean_1p0.nc'),
   'climate_std': PosixPath('/data/users/stephen.haddad/climate_zones/1961_1990/ensemble_std_1p0.nc'),
   'climate_zone': PosixPath('/data/users/stephen.haddad/climate_zones/1961_19

### Preparing a tabulated dataset
Currently the data is in gridded format in the netcdf files. For our simple machine learning model, we are going to be predicting the climtae zone for a particular point for 

We'll start by loading the three parts to the data for the historical period 1901 to 1930 and also for a future period 2017 to 2099. We load both of these because the data is lsightly different. For the historical period we have 1 simulation, whereas for the future period, we have different climate change scenarios, called [Shared Socioeconomic Pathways](https://en.wikipedia.org/wiki/Shared_Socioeconomic_Pathways), each represented in different climate stats and zones for the different pathways.

In [24]:
def create_dataframe_period(data_period_scenario, period, scenario):
    print(f'processing period {period} scenario {scenario}')
    df_mean = data_period_scenario['climate_mean'].to_dataframe().unstack(['time'])
    df_mean.columns = ['_'.join(map(str,l1))+ '_mean' for l1 in df_mean.columns]
    df_mean = df_mean.reset_index()
    
    df_std = data_period_scenario['climate_std'].to_dataframe().unstack(['time'])
    df_std.columns = ['_'.join(map(str,l1)) + '_std' for l1 in df_std.columns]
    df_std = df_std.reset_index()
    
    df_zone = data_period_scenario['climate_zone'].to_dataframe()
    df_zone = df_zone.reset_index()

    df_climate_zones = df_mean.merge(df_std, on=['lat','lon'])
    df_climate_zones = df_climate_zones.merge(df_zone, on=['lat','lon'])
    
    df_climate_zones = df_climate_zones[~df_climate_zones['precipitation_1.0_mean'].isna()]
    df_climate_zones['period_start'] = period[0]
    df_climate_zones['period_end'] = period[1] 
    df_climate_zones['scenario'] = scenario

    df_climate_zones['climate_group'] = df_climate_zones['kg_class'].apply(lambda v1: climate_group_lookup[int(v1)])
    df_climate_zones['climate_subgroup'] = df_climate_zones['kg_class'].apply(lambda v1: climate_group_lookup[int(v1)])
    
    return df_climate_zones

In [25]:
current_scenario = historic_scenario_str
current_period = list(time_periods.keys())[0]
df_period_scenario = create_dataframe_period(data_dict[current_period][current_scenario], current_period, current_scenario)
df_period_scenario

processing period (1901, 1930) scenario historic


,lat,lon,precipitation_1.0_mean,precipitation_2.0_mean,precipitation_3.0_mean,precipitation_4.0_mean,precipitation_5.0_mean,precipitation_6.0_mean,precipitation_7.0_mean,precipitation_8.0_mean,...,air_temperature_10.0_std,air_temperature_11.0_std,air_temperature_12.0_std,kg_class,kg_confidence,period_start,period_end,scenario,climate_group,climate_subgroup
2301,83.5,-38.5,12.1250,8.2500,8.9375,21.8750,15.5000,29.8750,35.8750,46.1250,...,2.5625,3.3125,3.6250,29.0,83.0,1901,1930,historic,E,E
2302,83.5,-37.5,10.8750,8.1250,8.6250,19.6250,13.8125,26.8750,32.0625,42.0000,...,3.0000,3.8125,4.1250,30.0,86.0,1901,1930,historic,E,E
2303,83.5,-36.5,9.9375,8.4375,8.8125,18.1250,12.3750,23.6875,28.5000,37.8125,...,2.5000,3.1875,3.4375,30.0,84.0,1901,1930,historic,E,E
2304,83.5,-35.5,10.0000,8.8750,9.0625,17.6250,12.0000,22.4375,26.9375,35.9375,...,2.3750,3.1250,3.4375,30.0,86.0,1901,1930,historic,E,E
2305,83.5,-34.5,10.1250,9.1875,9.4375,16.7500,11.3750,20.6250,24.9375,33.3750,...,1.9375,2.5000,2.8750,30.0,76.0,1901,1930,historic,E,E
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64795,-89.5,175.5,4.1250,4.8125,6.7500,7.0625,7.3750,5.9375,5.9375,9.0000,...,1.2500,2.0000,2.1250,30.0,99.0,1901,1930,historic,E,E
64796,-89.5,176.5,4.1250,4.8125,6.8125,7.0625,7.4375,5.9375,6.0000,9.1250,...,1.6875,2.2500,2.3125,30.0,99.0,1901,1930,historic,E,E
64797,-89.5,177.5,4.1875,4.8750,6.8125,7.1250,7.4375,6.0000,6.0000,9.3750,...,2.2500,2.6875,2.5625,30.0,99.0,1901,1930,historic,E,E
64798,-89.5,178.5,4.1875,4.8750,6.8750,7.1875,7.5000,6.0625,6.0625,9.6875,...,3.2500,3.3750,3.0625,30.0,99.0,1901,1930,historic,E,E


In [26]:
current_scenario = future_scenario_list[2]
current_period = list(time_periods.keys())[-1]
df_period_scenario = create_dataframe_period(data_dict[current_period][current_scenario], current_period, current_scenario)
df_period_scenario

processing period (2071, 2099) scenario ssp245


,lat,lon,precipitation_1.0_mean,precipitation_2.0_mean,precipitation_3.0_mean,precipitation_4.0_mean,precipitation_5.0_mean,precipitation_6.0_mean,precipitation_7.0_mean,precipitation_8.0_mean,...,air_temperature_10.0_std,air_temperature_11.0_std,air_temperature_12.0_std,kg_class,kg_confidence,period_start,period_end,scenario,climate_group,climate_subgroup
2301,83.5,-38.5,19.4375,13.5625,14.6875,28.6875,21.1250,32.3125,42.2500,54.3125,...,2.5000,2.2500,2.8750,29.0,92.0,2071,2099,ssp245,E,E
2302,83.5,-37.5,17.6875,13.5000,14.3750,25.8750,18.9375,29.0625,37.7500,49.9375,...,2.5000,2.1875,2.8750,29.0,86.0,2071,2099,ssp245,E,E
2303,83.5,-36.5,16.4375,14.1875,14.8125,24.1250,17.1250,25.5625,33.5625,45.3125,...,2.5000,2.1875,2.8750,29.0,84.0,2071,2099,ssp245,E,E
2304,83.5,-35.5,16.7500,14.9375,15.5000,23.6250,16.6250,24.2500,31.7500,43.5000,...,2.5000,2.1875,2.8750,30.0,80.0,2071,2099,ssp245,E,E
2305,83.5,-34.5,17.2500,15.6250,16.3125,22.6250,15.9375,22.3750,29.4375,40.8125,...,2.4375,2.1875,2.8750,29.0,82.0,2071,2099,ssp245,E,E
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64795,-89.5,175.5,3.6875,4.3750,6.1875,6.5000,6.6875,5.3750,5.3125,7.9375,...,0.8125,0.7500,0.8750,30.0,100.0,2071,2099,ssp245,E,E
64796,-89.5,176.5,3.6875,4.3750,6.2500,6.5625,6.7500,5.3750,5.3125,8.1250,...,0.8125,0.7500,0.9375,30.0,100.0,2071,2099,ssp245,E,E
64797,-89.5,177.5,3.7500,4.3750,6.2500,6.6250,6.8125,5.4375,5.3750,8.3750,...,0.8125,0.7500,1.0000,30.0,100.0,2071,2099,ssp245,E,E
64798,-89.5,178.5,3.7500,4.4375,6.2500,6.6250,6.8125,5.4375,5.3750,8.6250,...,0.8125,0.7500,1.0625,30.0,100.0,2071,2099,ssp245,E,E


In [27]:
for cp, ps in data_path_dict.items():
    for sid, pd in ps.items():
        for ds_id, ds_path in pd.items():
            print(cp, ds_id, ds_path.is_file())
data_path_dict

(1901, 1930) climate_mean True
(1901, 1930) climate_std True
(1901, 1930) climate_zone True
(1931, 1960) climate_mean True
(1931, 1960) climate_std True
(1931, 1960) climate_zone True
(1961, 1990) climate_mean True
(1961, 1990) climate_std True
(1961, 1990) climate_zone True
(1991, 2020) climate_mean True
(1991, 2020) climate_std True
(1991, 2020) climate_zone True
(2041, 2070) climate_mean True
(2041, 2070) climate_std True
(2041, 2070) climate_zone True
(2041, 2070) climate_mean True
(2041, 2070) climate_std True
(2041, 2070) climate_zone True
(2041, 2070) climate_mean True
(2041, 2070) climate_std True
(2041, 2070) climate_zone True
(2041, 2070) climate_mean True
(2041, 2070) climate_std True
(2041, 2070) climate_zone True
(2041, 2070) climate_mean True
(2041, 2070) climate_std True
(2041, 2070) climate_zone True
(2041, 2070) climate_mean True
(2041, 2070) climate_std True
(2041, 2070) climate_zone True
(2041, 2070) climate_mean True
(2041, 2070) climate_std True
(2041, 2070) climat

{(1901,
  1930): {'historic': {'climate_mean': PosixPath('/data/users/stephen.haddad/climate_zones/1901_1930/ensemble_mean_1p0.nc'),
   'climate_std': PosixPath('/data/users/stephen.haddad/climate_zones/1901_1930/ensemble_std_1p0.nc'),
   'climate_zone': PosixPath('/data/users/stephen.haddad/climate_zones/1901_1930/koppen_geiger_1p0.nc')}},
 (1931,
  1960): {'historic': {'climate_mean': PosixPath('/data/users/stephen.haddad/climate_zones/1931_1960/ensemble_mean_1p0.nc'),
   'climate_std': PosixPath('/data/users/stephen.haddad/climate_zones/1931_1960/ensemble_std_1p0.nc'),
   'climate_zone': PosixPath('/data/users/stephen.haddad/climate_zones/1931_1960/koppen_geiger_1p0.nc')}},
 (1961,
  1990): {'historic': {'climate_mean': PosixPath('/data/users/stephen.haddad/climate_zones/1961_1990/ensemble_mean_1p0.nc'),
   'climate_std': PosixPath('/data/users/stephen.haddad/climate_zones/1961_1990/ensemble_std_1p0.nc'),
   'climate_zone': PosixPath('/data/users/stephen.haddad/climate_zones/1961_19

In [33]:
data_path_dict[(2071,2099)]

{'ssp119': {'climate_mean': PosixPath('/data/users/stephen.haddad/climate_zones/2071_2099/ssp119/ensemble_mean_1p0.nc'),
  'climate_std': PosixPath('/data/users/stephen.haddad/climate_zones/2071_2099/ssp119/ensemble_std_1p0.nc'),
  'climate_zone': PosixPath('/data/users/stephen.haddad/climate_zones/2071_2099/ssp119/koppen_geiger_1p0.nc')},
 'ssp126': {'climate_mean': PosixPath('/data/users/stephen.haddad/climate_zones/2071_2099/ssp126/ensemble_mean_1p0.nc'),
  'climate_std': PosixPath('/data/users/stephen.haddad/climate_zones/2071_2099/ssp126/ensemble_std_1p0.nc'),
  'climate_zone': PosixPath('/data/users/stephen.haddad/climate_zones/2071_2099/ssp126/koppen_geiger_1p0.nc')},
 'ssp245': {'climate_mean': PosixPath('/data/users/stephen.haddad/climate_zones/2071_2099/ssp245/ensemble_mean_1p0.nc'),
  'climate_std': PosixPath('/data/users/stephen.haddad/climate_zones/2071_2099/ssp245/ensemble_std_1p0.nc'),
  'climate_zone': PosixPath('/data/users/stephen.haddad/climate_zones/2071_2099/ssp245

In [21]:
data_dict = {
    current_period: {
        scenario_id: {ds_id: xarray.open_dataset(current_path) for ds_id, current_path in ds_dict.items()} for scenario_id, ds_dict in data_path_dict[current_period].items()
    } for current_period in [(1901,1930), (2071,2099)]
}

In [37]:
climate_zones_df_list = []
for current_period, scenario_paths in data_path_dict.items():
    for current_scenario, scenario_data in scenario_paths.items():
        # load data for mthis poeriod/scenario
        current_data = {ds_id: xarray.open_dataset(current_path) for ds_id, current_path in scenario_data.items()}
        # process into a dataframe
        climate_zones_df_list += [create_dataframe_period(
            current_data, 
            current_period,
            current_scenario)]


processing period (1901, 1930) scenario historic
processing period (1931, 1960) scenario historic
processing period (1961, 1990) scenario historic
processing period (1991, 2020) scenario historic
processing period (2041, 2070) scenario ssp119
processing period (2041, 2070) scenario ssp126
processing period (2041, 2070) scenario ssp245
processing period (2041, 2070) scenario ssp370
processing period (2041, 2070) scenario ssp434
processing period (2041, 2070) scenario ssp460
processing period (2041, 2070) scenario ssp585
processing period (2071, 2099) scenario ssp119
processing period (2071, 2099) scenario ssp126
processing period (2071, 2099) scenario ssp245
processing period (2071, 2099) scenario ssp370
processing period (2071, 2099) scenario ssp434
processing period (2071, 2099) scenario ssp460
processing period (2071, 2099) scenario ssp585


In [44]:
climate_zones_merged_df = pandas.concat(climate_zones_df_list).reset_index().drop(['index'],axis='columns')
climate_zones_merged_df

,lat,lon,precipitation_1.0_mean,precipitation_2.0_mean,precipitation_3.0_mean,precipitation_4.0_mean,precipitation_5.0_mean,precipitation_6.0_mean,precipitation_7.0_mean,precipitation_8.0_mean,...,air_temperature_10.0_std,air_temperature_11.0_std,air_temperature_12.0_std,kg_class,kg_confidence,period_start,period_end,scenario,climate_group,climate_subgroup
0,83.5,-38.5,12.1250,8.2500,8.9375,21.8750,15.5000,29.8750,35.8750,46.1250,...,2.5625,3.3125,3.6250,29.0,83.0,1901,1930,historic,E,E
1,83.5,-37.5,10.8750,8.1250,8.6250,19.6250,13.8125,26.8750,32.0625,42.0000,...,3.0000,3.8125,4.1250,30.0,86.0,1901,1930,historic,E,E
2,83.5,-36.5,9.9375,8.4375,8.8125,18.1250,12.3750,23.6875,28.5000,37.8125,...,2.5000,3.1875,3.4375,30.0,84.0,1901,1930,historic,E,E
3,83.5,-35.5,10.0000,8.8750,9.0625,17.6250,12.0000,22.4375,26.9375,35.9375,...,2.3750,3.1250,3.4375,30.0,86.0,1901,1930,historic,E,E
4,83.5,-34.5,10.1250,9.1875,9.4375,16.7500,11.3750,20.6250,24.9375,33.3750,...,1.9375,2.5000,2.8750,30.0,76.0,1901,1930,historic,E,E
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
406489,-89.5,175.5,4.0000,4.5625,6.6250,7.0000,7.1875,5.7500,5.6250,8.3125,...,1.1250,1.1250,1.4375,30.0,100.0,2071,2099,ssp585,E,E
406490,-89.5,176.5,4.0000,4.5625,6.6875,7.0625,7.2500,5.8125,5.6250,8.4375,...,1.1250,1.1250,1.4375,30.0,100.0,2071,2099,ssp585,E,E
406491,-89.5,177.5,4.0625,4.6250,6.6875,7.1250,7.3125,5.8125,5.6875,8.7500,...,1.1250,1.1250,1.4375,30.0,100.0,2071,2099,ssp585,E,E
406492,-89.5,178.5,4.0625,4.6250,6.6875,7.1250,7.3125,5.8125,5.6875,9.0000,...,1.1250,1.1875,1.4375,30.0,100.0,2071,2099,ssp585,E,E


In [47]:
out_path = ml_ready_output_dir / csv_out_template.format(resolution=resolutions_dict[current_res])
out_path

PosixPath('/data/users/stephen.haddad/climate_zones/ml_ready/climate_zones_1p0.csv')

In [48]:
climate_zones_merged_df.to_csv(out_path)

### References
